# LINKS DESTE CONTEÚDO ONLINE
**Colab**: https://colab.research.google.com/drive/1VBdNj7S6NITOjdoTwCNl910thj89w2yu#scrollTo=akCNQfmmRFhV

**Pasta do Drive** https://drive.google.com/drive/u/0/folders/1Oe--XbZeteYnEnGYrvflC8uqWaZcNNzf

**GitHub**: https://github.com/g4br/geotec/

#Colab de Fonte de Dados Meteorológicos

Neste Colab, vamos trabalhar com os dados meteorologicos usando programação Python e eventualmente Bash (shell script)

## Meu primeiro comando em "shell"

Toda vez que eu for usar o "Linux" nessa maquina virtual, eu devo antecipar o comando o sinal de exclamação "!"

Os comando mais usados serão:

1. **!cd**: mudança de diretório/pasta
2. **!ls**: lista o que tem no diretório
3. **!mkdir**: cria diretório
4. **!rm**: remove arquivo / diretório
5. **!for**: loop / laço
6. **!wget**: comando para download de arquivos
7. **!echo**: escreve algo
8. **!mv**: move arquivos
9. **!sed**: um "editor de texto" via linha de comando


In [ ]:
!ls

### Comparando os comandos Bash dos comandos Python

In [ ]:
# Python
print('O curso de meteoro é o mais legal')

In [ ]:
# bash
!echo "Aí que delícia"

In [ ]:
# Aqui um laço em python
for i in range(1,10):
  print(i)

In [ ]:
# aqui um laço em Bash/Shell
!for a in `seq 1 10`; do echo $a; done

In [ ]:
# Existe um TRUQUE, caso eu precise fdazer algo um pouco mais complexo com bash
# E não quero ficar colocando "!" em todas as linhas....
# para isso eu uso na primeira linha o comando %%bash (isso é chamado de shebang)
# então... ↓↓

In [ ]:
%%bash
echo "aqui vou fazer vários 'echo'"
echo "e no final incluir um 'for'"

for ((i=1; i<=5; i++))
do
  echo $i
done

echo "a variável em Bash é chamada com o simbolo $<nome variável>"


## Montando o Drive nessa máquina virutal

Aqui monta o SEU drive nesta máquina virtural...
Assim, é possível criar e REMOVER (cuidadado com isso) arquivos do seu drive...

## A partir daqui, todos os arquivos estarão sendo gerados no SEU drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cd /content/drive/MyDrive

In [ ]:
!ls

In [ ]:
cd /content/drive/MyDrive/MaterialGeotec/basededados/

In [ ]:
ls

## Download da série de dados do INMET!

1. Banco de dados INMET: https://bdmep.inmet.gov.br/

2. Mapa de estações INMET: https://mapas.inmet.gov.br/

3. Consulta de estações INMET: https://tempo.inmet.gov.br/TabelaEstacoes/A001

### Contornando a restrição do servidor INMET - Erro 403
https://chatgpt.com/share/6996f460-0ad4-8001-9d59-4de429555933

**Bloqueio por “perfil de cliente”** no servidor (WAF / regras de segurança / anti-bot). No navegador, o pedido chega com um conjunto de headers e comportamento bem “browser-like” (User-Agent comum, Referer, cookies de sessão/consentimento, redirects, etc.). Já o wget chega com User-Agent: Wget/... e poucos headers — e muita infra coloca regra tipo “se for Wget/curl sem headers/cookies → 403”.

No caso do INMET, a própria página de “Dados Históricos Anuais” aponta os zips em https://portal.inmet.gov.br/uploads/dadoshistoricos/<ano>.zip (ex.: 2018) — mas o servidor pode exigir que o download “pareça” ter sido iniciado a partir dessa página.

In [ ]:
%%bash

# aqui vamos fazer um loop, contornando as travas do INMET e realizando o download
# dos dados com o comando "curl"
#
# Logo na sequência do download, faz o UNZIP para descompactar o dado original

for ((ano=2014; ano<=2025;ano++))
do
  curl -L \
    -A "Mozilla/5.0 (X11; Linux x86_64; rv:123.0) Gecko/20100101 Firefox/123.0" \
    -e "https://portal.inmet.gov.br/dadoshistoricos" \
    -o $ano.zip \
    "https://portal.inmet.gov.br/uploads/dadoshistoricos/$ano.zip"
  unzip $ano.zip
done

In [ ]:
# Aqui vamos instalar um "aplicativo" de linha de comando que ajusta todos os nomes de arquivos
!apt install detox

In [ ]:
!detox *.CSV

In [ ]:
!ls

In [ ]:
%%bash
# essa parte é somente para organizar os arquivos no meu driva
for ((ano=2014; ano<=2025;ano++))
do
  mkdir -p $ano
  mv *$ano*.CSV $ano
done

In [ ]:
import glob
arquivos = glob.glob('*/*A806*')
print(arquivos)

In [ ]:
import pandas as pd

df = pd.read_csv(arquivos[0], encoding='latin1', sep=';', decimal=',',skiprows=8)

In [ ]:
print(df)

In [ ]:
df.columns

In [ ]:
df[['DATA (YYYY-MM-DD)', 'HORA (UTC)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)']]

In [ ]:
df['datetime'] = pd.to_datetime(df['DATA (YYYY-MM-DD)'] + ' ' + df['HORA (UTC)'])

In [ ]:
df.set_index('datetime',inplace=True)

In [ ]:
df

In [ ]:
df_interesse = df[['DATA (YYYY-MM-DD)', 'HORA (UTC)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)']].copy()
df_interesse = df_interesse.loc['2014-03-01':'2014-03-31']

In [ ]:
df_interesse.rename(columns={'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)':'tmax', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'prec'}, inplace=True)

In [ ]:
df_interesse

In [ ]:
df_interesse.where(df_interesse != -9999, pd.NA, inplace=True)

In [ ]:
df_interesse

In [ ]:
# visualizando os dados
import matplotlib.pyplot as plt

df_interesse['tmax'].plot()
plt.show()

In [ ]:
df_interesse['prec'].plot()
plt.show()